### **1. Import des bibliothèques**

In [1]:
# Analyse et manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Prétraitement et métriques
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modèles
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb

# Sauvegarde
import joblib

# Interprétabilité
import shap


from pathlib import Path

DATASET_PATH = Path("C:/Users/LENOVO/Desktop/hackathon/HACKATHON-INDABAX-CAMEROON-2026-main/data/Dataset_complet_Meteo.csv")



# Suppression des warnings
import warnings
warnings.filterwarnings('ignore')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\LENOVO\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\LENOVO\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\LENOVO\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\LENOVO\anaconda3\Lib\site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

### **2. Chargement et analyse exploratoire (EDA)**

In [ ]:
# Charger les données
df = pd.read_csv('Dataset_complet_Meteo.csv', parse_dates=['time'], delimiter=";")
df = pd.read_csv(DATASET_PATH, parse_dates=['time'])
df.set_index('time', inplace=True)
df.sort_index(inplace=True)

# Aperçu
print("Shape:", df.shape)
df.head()

# Informations générales
df.info()

# Vérification des valeurs manquantes
missing = df.isnull().sum()
print(missing[missing > 0])

# Statistiques descriptives de la cible et des features
df[['et0_fao_evapotranspiration', 'temperature_2m_mean', 
    'shortwave_radiation_sum', 'wind_speed_10m_max', 
    'relative_humidity_2m_mean']].describe()

# Distribution de la cible
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(df['et0_fao_evapotranspiration'].dropna(), bins=50, kde=True)
plt.title('Distribution de ET0')
plt.subplot(1,2,2)
sns.boxplot(y=df['et0_fao_evapotranspiration'])
plt.title('Boxplot de ET0')
plt.tight_layout()
plt.show()

# Analyse temporelle : moyenne mensuelle par région
df['month'] = df.index.month
monthly_et0 = df.groupby(['region', 'month'])['et0_fao_evapotranspiration'].mean().unstack()
plt.figure(figsize=(12,6))
sns.heatmap(monthly_et0, cmap='YlOrRd', annot=True, fmt='.1f')
plt.title('ET0 mensuel moyen par région (mm/jour)')
plt.show()

# Corrélations avec la cible
features_meteo = ['temperature_2m_mean', 'shortwave_radiation_sum', 
                  'wind_speed_10m_max', 'relative_humidity_2m_mean']
corr_matrix = df[features_meteo + ['et0_fao_evapotranspiration']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Matrice de corrélation')
plt.show()

### **3. Feature engineering**

Bien que le document n’en demande pas explicitement, nous ajoutons des features temporelles et glissantes pour améliorer la prédiction (sans introduire de fuite future).

In [ ]:
# Nettoyage des valeurs manquantes (interpolation linéaire par ville)
df = df.groupby('city').apply(lambda group: group.interpolate(method='linear', limit_direction='both'))
df.dropna(subset=['et0_fao_evapotranspiration'], inplace=True)

# Features temporelles cycliques
df['day_of_year'] = df.index.dayofyear
df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
df['month'] = df.index.month
df['is_dry_season'] = df['month'].isin([11,12,1,2,3]).astype(int)

# Lags des features météo (1, 2, 3, 7 jours)
for lag in [1, 2, 3, 7]:
    for col in features_meteo:
        df[f'{col}_lag_{lag}'] = df.groupby('city')[col].shift(lag)

# Moyennes mobiles sur 3 et 7 jours
for window in [3, 7]:
    for col in features_meteo:
        df[f'{col}_roll{window}'] = df.groupby('city')[col].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )

# Supprimer les premières lignes devenues NaN à cause des lags
df.dropna(inplace=True)

print("Nouvelle taille après feature engineering :", df.shape)

### **4. Préparation des données pour l’apprentissage**

Nous utilisons un split temporel : toutes les données avant 2024 pour l’entraînement, 2024-2025 pour le test.
Cela respecte le caractère temporel des séries

In [ ]:
# Définir la liste complète des features (on exclut les identifiants et la cible)
exclude_cols = ['city', 'region', 'latitude', 'longitude', 'et0_fao_evapotranspiration', 
                'weather_code', 'rain_sum', 'precipitation_sum']  # precipitation_sum est redondante
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df['et0_fao_evapotranspiration']

# Split temporel (2020-2023 train, 2024-2025 test)
train_mask = df.index.year < 2024
test_mask = df.index.year >= 2024

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train : {X_train.shape}, Test : {X_test.shape}")

# Normalisation (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Sauvegarde du scaler
joblib.dump(scaler, 'scaler_eto.pkl')

### **5. Entraînement de plusieurs modèles et sélection du meilleur**

Nous testons trois modèles : Ridge (linéaire), Random Forest, XGBoost.
La métrique de sélection est la MAE (Mean Absolute Error).

In [ ]:
models = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
}

results = {}
best_model = None
best_mae = float('inf')

for name, model in models.items():
    print(f"Entraînement de {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name} -> MAE: {mae:.3f}, RMSE: {rmse:.3f}, R2: {r2:.3f}\n")
    
    if mae < best_mae:
        best_mae = mae
        best_model = model
        best_model_name = name

print(f"Meilleur modèle : {best_model_name} (MAE = {best_mae:.3f})")

### **Comparaison visuelle des performances**

In [ ]:
results_df = pd.DataFrame(results).T
results_df.plot(kind='bar', figsize=(10,5))
plt.title('Comparaison des modèles pour la prédiction ET0')
plt.ylabel('Erreur / Score')
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.show()

### **6. Sauvegarde du meilleur modèle**

In [ ]:
# Sauvegarde du modèle et des métadonnées
joblib.dump(best_model, 'best_eto_model.pkl')

# Sauvegarde de la liste des features (utile pour l'inférence)
with open('eto_features.txt', 'w') as f:
    for col in feature_cols:
        f.write(col + '\n')

print("Modèle et scaler sauvegardés.")

### **7. Interprétabilité**

Nous utilisons SHAP pour expliquer les prédictions du meilleur modèle (ici XGBoost ou RandomForest).

In [ ]:
# Vérifier que le meilleur modèle est de type arbre (sinon, adapter)
if best_model_name in ['RandomForest', 'XGBoost']:
    # Prélever un sous-ensemble pour l'explication (plus rapide)
    X_sample = X_test_scaled[:200]
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_sample)
    
    # Résumé global
    shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
    plt.title(f"SHAP summary plot - {best_model_name}")
    plt.tight_layout()
    plt.show()
    
    # Force plot pour une prédiction individuelle
    shap.initjs()
    shap.force_plot(explainer.expected_value, shap_values[0,:], 
                    X_sample[0,:], feature_names=feature_cols, matplotlib=True)
else:
    # Pour Ridge, on peut simplement afficher les coefficients
    coefs = pd.Series(best_model.coef_, index=feature_cols).sort_values()
    coefs.plot(kind='barh', figsize=(10,6))
    plt.title('Coefficients du modèle Ridge')
    plt.xlabel('Valeur du coefficient')
    plt.grid()
    plt.show()

### Analyse supplémentaire : importance des features (pour modèles arborescents)

In [ ]:
if best_model_name in ['RandomForest', 'XGBoost']:
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
    plt.figure(figsize=(10,6))
    feat_imp.head(15).plot(kind='bar')
    plt.title(f'Importance des features - {best_model_name}')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.show()

### **8. Conclusion et pistes d’amélioration**

In [ ]:
print("""
Résumé :
- La prédiction de l'ETO est réalisable avec une erreur absolue moyenne d'environ X mm/jour.
- Les features les plus importantes sont : (à compléter selon sortie SHAP).
- Le modèle sauvegardé peut être utilisé pour inférer l'ETO à partir des seules variables météo de base.

Pistes d'amélioration :
- Ajouter des données de vent à différentes hauteurs.
- Utiliser des modèles séquentiels (LSTM) pour capter les dépendances temporelles longues.
- Intégrer des indices de végétation (NDVI) si disponibles.
""")

### **9. Exemple d’inférence avec le modèle sauvegardé**

In [ ]:
# Chargement du modèle et du scaler
loaded_model = joblib.load('best_eto_model.pkl')
loaded_scaler = joblib.load('scaler_eto.pkl')

# Exemple : prédire l'ETO pour une nouvelle journée (avec les mêmes features)
new_data = pd.DataFrame({
    'temperature_2m_mean': [28.5],
    'shortwave_radiation_sum': [22.0],
    'wind_speed_10m_max': [3.2],
    'relative_humidity_2m_mean': [65.0],
    # ... toutes les autres features doivent être présentes (lags, rolling, etc.)
})
# Il faudrait recréer les mêmes transformations (lags nécessitent historique)
# Pour un usage réel, on recommande de créer une classe pipeline complète.

print("Prédiction ET0 :", loaded_model.predict(loaded_scaler.transform(new_data))[0])